# Lección 3: Filtrado, ordenamiento y agrupación de datos

**Aprendiz:** Carlos Eduardo Herrera Sibaja
**Programa:** Análisis y Desarrollo de Software
**Evidencia:** GA1-220501094-01-AA1-EV06

Las tablas limpias se convierten en respuestas: cuándo compran los clientes y con qué frecuencia vuelven. **Filtrar** elige las filas que importan, **ordenar** las jerarquiza y **agrupar** las resume. Cada pregunta se traduce primero a la forma **"métrica por grupo"** o a una **condición**: "¿a qué hora se compra más?" se convierte en "número de pedidos por hora".

**Entradas:** `orders_clean.parquet` y `order_products_clean.parquet`, generados por [`lesson_2_data_cleaning.ipynb`](lesson_2_data_cleaning.ipynb).
**Salidas:** resúmenes por día, segmento de cliente y franja horaria en `data/processed/`.

El notebook forma parte de los laboratorios de Pandas del proyecto Instacart; el índice y el orden de ejecución están en [`pandas_labs.ipynb`](pandas_labs.ipynb). Ejecútelo completo con **Run All**.

## 3.0 Preparación del entorno y carga de los datos

Se cargan las tablas limpias de la Lección 2. Parquet conserva los tipos optimizados (`int8`, `int16`, `Int16`, etc.), así que no hace falta repetir la limpieza.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

SALIDA = Path('../data/processed')
necesarios = ['orders_clean.parquet', 'order_products_clean.parquet']
faltan = [archivo for archivo in necesarios if not (SALIDA / archivo).exists()]
if faltan:
    raise FileNotFoundError(f'Ejecute primero lesson_2_data_cleaning.ipynb. Faltan: {faltan}')

orders      = pd.read_parquet(SALIDA / 'orders_clean.parquet')
order_prods = pd.read_parquet(SALIDA / 'order_products_clean.parquet')

print('pandas', pd.__version__)
print('orders     ', orders.shape)
print('order_prods', order_prods.shape)

pandas 3.1.0
orders      (478952, 6)
order_prods (4545007, 4)


## 3.1 Ejemplos de la lección

### Filtrado con máscaras booleanas

In [2]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


In [3]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))

45599


### Filtrado con `query`

In [4]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))

1449


### Ordenamiento con `sort_values` y `nlargest`

In [5]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100
        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


### Agrupación con `groupby`

In [6]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())

10 40578


In [7]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


### Tabla dinámica con `pivot_table`

In [8]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


### Valores por fila con `transform`

In [9]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')

231946 30527


## 3.2 Laboratorio guiado 3: perfil de hábitos de compra

Encargo de mercadeo: diseñar campañas según qué tan seguido compra cada cliente y saber a qué hora compra cada tipo de cliente. Plan: contar pedidos por cliente, segmentar clientes, clasificar pedidos en franjas horarias y cruzar segmentos con franjas.

### Paso 1: pedidos por cliente

In [10]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

### Paso 2: segmentar clientes con `pd.cut`

In [11]:
segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))

order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


### Paso 3: llevar el segmento a cada pedido con `map`

In [12]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


### Paso 4: resumen por segmento

In [13]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


**Interpretación:** el 19,4 % de los clientes (los frecuentes) hace casi la mitad de los pedidos (231 946 de 478 952) y vuelve con una mediana de 6 días; los regulares vuelven cada 11 días y los ocasionales cada 16. Los ocasionales tienen espera porque su único pedido en estos datos no es necesariamente el primero que hicieron en la tienda.

### Paso 5: franjas horarias y tabla cruzada

In [14]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


**Interpretación:** todos los segmentos compran sobre todo en la tarde, pero los frecuentes compran más en la mañana (35,2 % frente a 30,0 % de los ocasionales) y algo menos en la tarde y la noche. Para mercadeo, una campaña de fidelización dirigida a clientes frecuentes debería enviarse por la mañana.

### Paso 6: las combinaciones de día y franja más activas

In [15]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

### Paso 7: guardar los resúmenes y dejar `orders` como estaba

In [16]:
resumen_dia.to_csv(SALIDA / 'orders_by_weekday.csv')
resumen_segmento.to_csv(SALIDA / 'customer_segments.csv')
perfil.to_csv(SALIDA / 'segment_time_profile.csv')

orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)

(478952, 6)


**Conclusiones clave del laboratorio 3**
- Una pregunta de negocio compleja se resuelve en pasos: agrupar, categorizar, devolver al detalle y cruzar.
- `pd.cut` convierte números en categorías con límites explícitos.
- `map` lleva un resultado calculado por grupo a cada fila del detalle.
- `crosstab` con `normalize` permite comparar grupos de distinto tamaño.

## 3.3 Reto 3

**Ejercicio 1.** ¿Cuáles son las tres horas con más pedidos los domingos y cuántos pedidos tiene cada una?

*Condición + métrica por grupo:* pedidos con `order_dow == 0` → número de pedidos por hora → las tres mayores.

In [17]:
domingos = orders[orders['order_dow'] == 0]
domingos.groupby('order_hour_of_day').size().nlargest(3).rename('pedidos')

order_hour_of_day
15    7585
14    7533
13    7483
Name: pedidos, dtype: int64

**Interpretación:** los domingos el pico es a primera hora de la tarde: 15 h (7 585 pedidos), 14 h (7 533) y 13 h (7 483). Las tres horas tienen volúmenes casi iguales, así que el domingo la demanda alta es una franja continua, más tardía que el pico general de la semana (10 a. m.).

**Ejercicio 2.** ¿A qué hora es mayor la espera media entre pedidos y a qué hora es menor?

*Métrica por grupo:* media de `days_since_prior_order` por hora.

In [18]:
espera_por_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean()
print(f'Mayor espera: {espera_por_hora.idxmax()} h ({espera_por_hora.max():.2f} días)')
print(f'Menor espera: {espera_por_hora.idxmin()} h ({espera_por_hora.min():.2f} días)')

Mayor espera: 0 h (12.52 días)
Menor espera: 22 h (10.62 días)


**Interpretación:** la mayor espera media está a las 0 h (12,52 días) y la menor a las 22 h (10,62 días). La diferencia es de unos 2 días: quien compra a medianoche tiende a ser un cliente menos regular, mientras que las compras de las 22 h vienen de clientes que vuelven más seguido.

**Ejercicio 3.** ¿Cuántos clientes tienen 10 pedidos o más?

*Métrica por grupo + condición:* número de pedidos por cliente ≥ 10.

In [19]:
pedidos_por_cliente = orders.groupby('user_id').size()
print('Clientes con 10 pedidos o más:', (pedidos_por_cliente >= 10).sum())

Clientes con 10 pedidos o más: 6327


**Interpretación:** 6 327 clientes tienen 10 pedidos o más, apenas el 4 % de los 157 437 clientes. Es un núcleo pequeño de clientes muy fieles, útil para estudiar hábitos de recompra.

**Ejercicio 4.** ¿Cuántos pedidos se hicieron exactamente 7 días después del anterior y qué porcentaje del total representan?

*Condición:* `days_since_prior_order == 7`.

In [20]:
a_los_7 = (orders['days_since_prior_order'] == 7).sum()
print('Pedidos a los 7 días:', a_los_7)
print(f'Porcentaje del total: {a_los_7 / len(orders):.2%}')
print('Valor más frecuente de espera:', orders['days_since_prior_order'].mode()[0], 'días')

Pedidos a los 7 días: 44577
Porcentaje del total: 9.31%
Valor más frecuente de espera: 30 días


**Interpretación:** 44 577 pedidos (9,31 % del total) se hicieron exactamente una semana después del anterior. Es uno de los picos de la distribución: muchos clientes hacen la compra semanal el mismo día. El valor más frecuente es 30 días, pero ese pico está inflado porque 30 es el tope de la columna (agrupa "30 días o más").

**Ejercicio 5.** Usando `order_prods`: ¿cuántos pedidos tienen 20 productos o más? ¿Cuál es el pedido más grande y cuántos productos tiene?

*Métrica por grupo:* número de productos por pedido.

In [21]:
productos_por_pedido = order_prods.groupby('order_id').size()
print('Pedidos con 20 productos o más:', (productos_por_pedido >= 20).sum())
print('Pedido más grande:', productos_por_pedido.idxmax(),
      'con', productos_por_pedido.max(), 'productos')

Pedidos con 20 productos o más: 49420
Pedido más grande: 61355 con 127 productos


**Interpretación:** 49 420 pedidos tienen 20 productos o más (el 11 % de los 450 046 pedidos con productos). El pedido más grande es el 61355, con 127 productos, muy por encima del límite de 64 posiciones del carrito que explica los nulos del Reto 2.

**Ejercicio 6.** Usando `order_prods`: tasa de recompra (media de `reordered`) para las posiciones 1 a 5 del carrito. ¿Qué patrón se ve?

*Condición + métrica por grupo:* posiciones 1 a 5 → media de `reordered` por posición.

In [22]:
primeras_posiciones = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
primeras_posiciones.groupby('add_to_cart_order')['reordered'].mean().round(3).rename('tasa_recompra')

add_to_cart_order
1    0.679
2    0.677
3    0.659
4    0.638
5    0.617
Name: tasa_recompra, dtype: float64

**Interpretación:** la tasa de recompra baja a medida que avanza el carrito: 0,679 en la posición 1 y 0,617 en la 5. Los clientes agregan primero los productos que ya conocen y compran siempre, y dejan para después los productos nuevos o de prueba.

**Ejercicio 7.** Comparar la mediana de `days_since_prior_order` de los pedidos de fin de semana (domingo y sábado) con la del resto de la semana.

*Métrica por grupo:* mediana de la espera según la máscara `isin([0, 6])`.

In [23]:
fin_de_semana = orders['order_dow'].isin([0, 6]).map({True: 'fin de semana', False: 'entre semana'})
orders.groupby(fin_de_semana)['days_since_prior_order'].agg(['median', 'mean', 'count']).round(2)

,median,mean,count
order_dow,,,
entre semana,7.0,10.87,312767
fin de semana,8.0,11.62,137368


**Interpretación:** la mediana de espera es de 8 días para los pedidos de fin de semana y de 7 días entre semana (medias de 11,62 y 10,87). Quienes compran entre semana vuelven un poco antes, lo que encaja con la compra semanal en un día laboral fijo.

---
Siguiente: [`lesson_4_data_merging.ipynb`](lesson_4_data_merging.ipynb).